In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.impute import SimpleImputer

In [ ]:
clin_file = "All_Subjects_DXSUM_26Aug2025_clinical_patient_data.csv"
nda_file  = "ADNI3_PLINK_FINAL_2nd_RAW.csv"
out_csv   = "ADNI3_DNA.csv"

# Select key clinical columns
clin_cols = ["PTID", "RID", "VISCODE", "EXAMDATE", "PHASE", "DIAGNOSIS"]
dfc = pd.read_csv(clin_file, usecols=clin_cols, low_memory=False)

# Keep only samples with valid diagnosis information
dfc = dfc[~dfc["DIAGNOSIS"].isna()].copy()

# Load DNA data without renaming columns
df_nda = pd.read_csv(nda_file, low_memory=False)

# Merge clinical and DNA data (RID from clinical matches IID from DNA)
df_merged = pd.merge(dfc, df_nda, left_on="PTID", right_on="IID", how="inner")

# Save the merged dataset
df_merged.to_csv(out_csv, index=False)
print("Merge completed. Saved to:", out_csv)

✅ Merge completed. Saved to: /content/drive/MyDrive/5703/FL/ADNI3_DNA.csv


In [ ]:
print(df_nda.columns[:20])

In [ ]:
def clean_adni_genetic_data(file_path, output_path, var_keep=20000, missing_threshold=0.3):
    print(f"Loading data from: {file_path}")
    df = pd.read_csv(file_path, low_memory=False)
    print("Original shape:", df.shape)

    # Extract metadata columns
    meta_cols = ["PTID", "RID", "VISCODE", "EXAMDATE", "PHASE", "DIAGNOSIS"]
    meta_cols = [c for c in meta_cols if c in df.columns]
    meta = df[meta_cols].copy()

    # Extract SNP numeric features
    snp_cols = [c for c in df.columns if c not in meta_cols]
    numeric = df[snp_cols].select_dtypes(include=[np.number])

    print(f"Metadata cols: {len(meta_cols)}, SNP cols: {numeric.shape[1]}")

    del df
    gc.collect()

    # Remove high-missing features
    missing_ratio = numeric.isna().mean()
    keep_cols = missing_ratio[missing_ratio <= missing_threshold].index
    numeric = numeric[keep_cols]
    print(f"After dropping high-missing features ({missing_threshold*100}% threshold): {numeric.shape}")

    # Fill missing values with median
    numeric[:] = SimpleImputer(strategy="median").fit_transform(numeric)
    print("Missing values imputed with median")

    # Convert to float32 for memory efficiency
    numeric = numeric.astype(np.float32)
    print("Converted to float32 (memory reduced)")

    # Variance-based feature selection
    variances = numeric.var().sort_values(ascending=False)
    top_features = variances.head(var_keep).index
    numeric = numeric[top_features]
    print(f"After variance filtering (Top {var_keep} features): {numeric.shape}")

    # Merge and save
    df_cleaned = pd.concat([meta, numeric], axis=1)
    print("Final cleaned shape:", df_cleaned.shape)

    # Free memory
    del numeric, meta
    gc.collect()

    df_cleaned.to_csv(output_path, index=False)
    print(f"Cleaned data saved to: {output_path}")

    return df_cleaned

clean_adni_genetic_data(
    file_path="ADNI3_DNA.csv",
    output_path="ADNI3_DNA_cleaned.csv"
)


In [ ]:
file_path = "ADNI3_DNA_cleaned.csv"

df_head = pd.read_csv(file_path, nrows=5)

print(df_head.head())

前 5 行数据：
         PTID   RID VISCODE    EXAMDATE  PHASE  DIAGNOSIS  MTReverseND5_14_T  \
0  073_S_0909   909      bl  2006-11-27  ADNI1        2.0                2.0   
1  130_S_1201  1201      bl  2007-02-02  ADNI1        3.0                2.0   
2  073_S_0909   909     m06  2007-07-06  ADNI1        2.0                2.0   
3  130_S_1201  1201     m06  2007-08-03  ADNI1        3.0                2.0   
4  073_S_0909   909     m12  2007-11-20  ADNI1        2.0                2.0   

   MitoT12706C_T  rs3937033_T  rs2853826_G  ...  GSA-rs132075_T  rs1146224_C  \
0            2.0          2.0          2.0  ...             0.0          1.0   
1            2.0          2.0          2.0  ...             0.0          1.0   
2            2.0          2.0          2.0  ...             0.0          1.0   
3            2.0          2.0          2.0  ...             0.0          1.0   
4            2.0          2.0          2.0  ...             0.0          1.0   

   rs7253468_C  rs1806487_C  